# Is the Judge the Bottleneck? Build the HotpotQA search index on Kaggle

This notebook builds the **BM25 search index** over HotpotQA's 2017 Wikipedia (about 5.2 million intro paragraphs),
using the project's own build code, `build_index_if_missing()` in `pipeline/retriever.py`, from the GitHub branch
`SC.PV2`. It is only needed if the index has to be rebuilt: the current index is already the Kaggle dataset
[`shoimyachy/hotpotqa-bm25-index`](https://www.kaggle.com/datasets/shoimyachy/hotpotqa-bm25-index).

**Why Kaggle:** Stanford's server, which hosts the Wikipedia download, refuses connections from the Mac; Kaggle
reached it when the current index was built (Oct 4).

**Before you run it**
1. On Kaggle: **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. In the right-hand panel (**Settings**): **Accelerator: None** (building is CPU work, so no GPU hours are used) and
   **Internet: On** (Internet needs a phone-verified Kaggle account).
3. **Save Version → Save & Run All (Commit)**. The notebook then runs in the background, even with the browser closed.

**What it does:** downloads Wikipedia (1.55 GB) into `data/wiki/`, unpacks it, and builds the index into
`data/wiki/bm25_index/` (about 2.8 GB). The download resumes by itself after a dropped connection, and a half-finished
unpack or build is redone, never mistaken for a finished one. Build time depends on the session; expect a long run.

## 1. Settings

Change only the lines in this cell.

In [ ]:
# Here is where the code comes from.
REPOSITORY_ADDRESS = "https://github.com/shoimya/Is-the-Judge-the-Bottleneck.git"
BRANCH = "SC.PV2"

## 2. Check the machine

Without internet nothing can be downloaded, so the notebook stops here with a clear message.

In [ ]:
import os
import socket
import subprocess
import sys
from pathlib import Path

# Here we tell the machines apart the same way pipeline/settings.py does: Kaggle sets KAGGLE_KERNEL_RUN_TYPE.
RUNNING_ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print("Machine:", "Kaggle" if RUNNING_ON_KAGGLE else "this computer")


def check_the_internet_is_on() -> None:
    """Stop with a clear message if this machine can't reach the internet."""
    # Here we look up github.com; if that fails, nothing can be downloaded.
    try:
        socket.gethostbyname("github.com")
        print("Internet: on")
    except OSError:
        raise RuntimeError("No internet. On Kaggle: verify your phone number (kaggle.com > Settings), "
                           "then switch Internet on in the notebook's Settings and run again.")


check_the_internet_is_on()

## 3. Get the project code

On Kaggle this downloads branch `BRANCH` from GitHub (or, if it is already here, resets it to exactly what GitHub has),
then installs the project's libraries. On the Mac it uses the repo this notebook sits in.

In [ ]:
KAGGLE_PROJECT_FOLDER = Path("/kaggle/working/Is-the-Judge-the-Bottleneck")


def find_or_clone_project_folder() -> Path:
    """On Kaggle, clone the project (or reset it to GitHub's version); on the Mac, find the repo above this notebook."""
    if not RUNNING_ON_KAGGLE:
        # Here we go up from the notebook's folder until we reach the folder that holds config.yaml.
        for folder in [Path.cwd(), *Path.cwd().parents]:
            if (folder / "config.yaml").exists():
                return folder
        raise FileNotFoundError("No project folder (a folder with config.yaml) was found above this notebook.")

    if KAGGLE_PROJECT_FOLDER.exists():
        # Here we reset an existing copy to exactly what GitHub has on BRANCH, throwing away any stray edits.
        subprocess.run(["git", "-C", str(KAGGLE_PROJECT_FOLDER), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(KAGGLE_PROJECT_FOLDER), "checkout", "--force", "-B", BRANCH,
                        f"origin/{BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY_ADDRESS, str(KAGGLE_PROJECT_FOLDER)], check=True)
    return KAGGLE_PROJECT_FOLDER


PROJECT_FOLDER = find_or_clone_project_folder()
os.chdir(PROJECT_FOLDER)

# Here, on Kaggle, we install the exact library versions the project needs (the Mac's .venv already has them).
if RUNNING_ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Here we let Python import the project's code (pipeline/) from the project folder straight away.
sys.path.insert(0, str(PROJECT_FOLDER))

# Here we print which commit of the code this run uses, so the index can be traced back to the code that built it.
last_commit = subprocess.run(["git", "log", "-1", "--format=%h %s (%ci)"], capture_output=True, text=True)
print(f"Project folder: {PROJECT_FOLDER}")
print(f"Code version:   {last_commit.stdout.strip()}")

## 4. Build the index

This calls the project's own `build_index_if_missing()`. If `data/wiki/bm25_index/` already exists it says so and
does nothing (delete that folder to build a fresh one). Otherwise it downloads and unpacks Wikipedia into `data/wiki/`,
builds the index in two passes, and saves it. A failed step stops this cell with its own error.

In [ ]:
from pipeline.retriever import build_index_if_missing, index_dir

print(f"Index destination: {index_dir()}", flush=True)
build_index_if_missing()

## How to use the index

When the notebook has finished, the index is in this notebook's output, at
`Is-the-Judge-the-Bottleneck/data/wiki/bm25_index/` (7 files, about 2.8 GB, starting with `params.index.json`).
The output also holds the Wikipedia download under `data/wiki/` (about 3 GB more), which nothing needs once the index
is built.

1. **Save it as a Kaggle dataset.** On the notebook's page, open the **Output** tab and choose **New Dataset**, and keep
   it private. The current one is
   [`shoimyachy/hotpotqa-bm25-index`](https://www.kaggle.com/datasets/shoimyachy/hotpotqa-bm25-index); give a rebuilt
   one a new name rather than replacing it.
2. **Use it on Kaggle.** In `Notebooks/kaggle_methods.ipynb`: **Add Input → Datasets →** your index dataset. Its cell 6
   finds the index by its `params.index.json` (wherever it sits in the dataset), links it into
   `data/wiki/bm25_index/`, and checks that a plain search of every Pilot question gives exactly the Mac's results
   (`results/pilot/bm25_recall.csv` unchanged). A rebuilt index must pass that check before it is used.
3. **Use it on the Mac.** Download the dataset from its Kaggle page and unzip it so the 7 files sit directly in
   `data/wiki/bm25_index/`. Then press Run on `project_check.py` (the search index check must pass) and on
   `pipeline/retriever.py` (it searches with a Pilot question and prints the results).
4. **In code**, open it once and search with it:
   ```python
   from pipeline.retriever import Retriever
   retriever = Retriever.load()
   results = retriever.search("Which river flows through Budapest?", k=5)
   ```